# Análise Comparativa de Desempenho: Ingestão Batch vs. Stream — v2

**Workloads:** 1x · 2x · 4x &emsp; **Execuções por workload:** 3  
**Objetivo:** (1) verificar consistência intra-workload entre repetições; (2) comparar os paradigmas batch/stream entre workloads.  
**Formato v2:** gráficos lado a lado — esquerda = batch, direita = stream; sem escala logarítmica.

## 0. Setup

In [ ]:

import json, glob, os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')

# ── Estilo acadêmico IEEE ────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'serif',
    'font.size':          9,
    'axes.titlesize':    10,
    'axes.labelsize':     9,
    'xtick.labelsize':    8,
    'ytick.labelsize':    8,
    'legend.fontsize':    8,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'savefig.format':    'png',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.35,
    'grid.linestyle':    '--',
})

# Paleta Paul Tol – daltônica-segura
C = {'1x': '#4477AA', '2x': '#EE6677', '4x': '#228833',
     'batch': '#4477AA', 'stream': '#EE6677'}

# Tamanhos IEEE (polegadas)
FIG1  = (3.54, 2.6)   # coluna única
FIG2  = (7.16, 3.0)   # duas colunas
FIG2T = (7.16, 4.5)   # duas colunas, mais alto
FIG3  = (7.16, 5.5)   # tall

RUNS_DIR  = 'runs'
FIGS_DIR  = 'Figs/comparativo_v2'
os.makedirs(FIGS_DIR, exist_ok=True)
print('Setup OK')


## 1. Carregamento de dados

In [2]:
# ── Inventário de runs ────────────────────────────────────────────────────────
WORKLOADS = ['1x', '2x', '4x']

def _discover_runs():
    meta = []
    for wl in WORKLOADS:
        dirs = sorted(glob.glob(f'{RUNS_DIR}/experimento_{wl}_*'))
        for i, rdir in enumerate(dirs, 1):
            rdir = rdir.replace(chr(92), '/')
            has_sum = bool(glob.glob(f'{rdir}/metrics/summary_*.json'))
            meta.append((wl, f'R{i}', rdir, has_sum))
    return meta

RUN_META = _discover_runs()
print('Runs descobertos:')
for _wl, _rid, _rdir, _hs in RUN_META:
    _tag = 'sim' if _hs else 'nao'
    print(f'  {_wl} {_rid}: {_rdir}  (summary: {_tag})')

def parse_bytes_mib(s):
    m = re.match(r'([\d.]+)\s*(B|kB|MB|MiB|GiB|GB)', str(s).strip(), re.I)
    if not m: return np.nan
    v, u = float(m.group(1)), m.group(2).lower()
    return v * {'b':1/1048576,'kb':1/1024,'mb':1,'mib':1,'gib':1024,'gb':953.674}[u]

# ── Summaries ─────────────────────────────────────────────────────────────────
summaries = []
for wl, rid, rdir, has_sum in RUN_META:
    if not has_sum: continue
    jf = glob.glob(f'{rdir}/metrics/summary_*.json')
    if not jf: continue
    with open(jf[0], encoding='utf-8') as f:
        d = json.load(f)
    d.update({'workload': wl, 'run_id': rid, 'run_dir': rdir})
    summaries.append(d)

# ── Batch throughput ──────────────────────────────────────────────────────────
tp_frames = []
for wl, rid, rdir, has_sum in RUN_META:
    if not has_sum: continue
    tf = glob.glob(f'{rdir}/metrics/batch_throughput_*.csv')
    if not tf: continue
    df = pd.read_csv(tf[0]).drop_duplicates(subset=['batch_id'])
    df['workload'], df['run_id'] = wl, rid
    tp_frames.append(df)
df_tp = pd.concat(tp_frames, ignore_index=True) if tp_frames else pd.DataFrame()

# ── Docker stats ──────────────────────────────────────────────────────────────
dock_frames = []
for wl, rid, rdir, _ in RUN_META:
    df = glob.glob(f'{rdir}/metrics/docker_stats_*.csv')
    if not df: continue
    d = pd.read_csv(df[0])
    d['workload'], d['run_id'] = wl, rid
    d['timestamp'] = pd.to_datetime(d['timestamp'], utc=True, errors='coerce')
    d['cpu_pct'] = d['cpu_perc'].str.rstrip('%').astype(float)
    d['mem_mib'] = d['mem_usage'].str.split('/').str[0].apply(parse_bytes_mib)
    t0 = d['timestamp'].min()
    d['t_min'] = (d['timestamp'] - t0).dt.total_seconds() / 60
    dock_frames.append(d[['timestamp','t_min','workload','run_id','name','cpu_pct','mem_mib']])
df_dock = pd.concat(dock_frames, ignore_index=True) if dock_frames else pd.DataFrame()

# Restringe docker stats aos 60 min de experimento (exclui buffer pós-gerador)
df_dock = df_dock[df_dock['t_min'] <= 60.0].copy()

print(f'Summaries: {len(summaries)}  |  Batches: {len(df_tp)}  |  Docker rows: {len(df_dock):,}')
print('Containers:', sorted(df_dock['name'].unique()))

Runs descobertos:
  1x R1: runs/experimento_1x_2026_07_01_20_25  (summary: sim)
  1x R2: runs/experimento_1x_2026_07_01_23_15  (summary: sim)
  1x R3: runs/experimento_1x_2026_07_02_23_42  (summary: sim)
  2x R1: runs/experimento_2x_2026_07_01_01_03  (summary: sim)
  2x R2: runs/experimento_2x_2026_07_01_09_28  (summary: sim)
  2x R3: runs/experimento_2x_2026_07_01_19_11  (summary: sim)
  4x R1: runs/experimento_4x_2026_06_30_19_55  (summary: sim)
  4x R2: runs/experimento_4x_2026_06_30_23_27  (summary: nao)
  4x R3: runs/experimento_4x_2026_06_30_23_30  (summary: sim)
Summaries: 8  |  Batches: 571  |  Docker rows: 118,721
Containers: ['batch_consumer', 'grafana', 'influxdb', 'mosquitto', 'postgres', 'sensor_generator', 'stream_consumer']


In [3]:
# ── Tabela de summaries flat ──────────────────────────────────────────────────
rows = []
for s in summaries:
    pg, inf_, sc, bt = s['postgres'], s['influxdb'], s['stream_consumer'], s['batch_throughput']
    recv  = sc['contador']
    infl  = inf_['total_records']
    loss  = (recv - infl) / recv * 100 if recv else 0
    rows.append({
        'workload':          s['workload'],
        'run_id':            s['run_id'],
        'pg_records':        pg['total_records'],
        'batch_tp_rps':      bt['avg_throughput_records_per_s'],
        'lat_avg_s':         pg['latency_avg_ms'] / 1000,
        'lat_p50_s':         pg['latency_p50_ms'] / 1000,
        'lat_p95_s':         pg['latency_p95_ms'] / 1000,
        'stream_recv':       recv,
        'stream_influx':     infl,
        'batch_effective_rps': round(pg['total_records'] / 3600, 1),
        'stream_rps':        round(infl / 3600, 1),
        'stream_loss_pct':   round(loss, 2),
    })
df_sum = pd.DataFrame(rows)

print('⚠  4x R1 (jun-10): pg_records contém dados de run anterior (sem truncate).\n'
      '   Throughput batch e docker stats são válidos; pg_records NÃO comparável.\n')

display(
    df_sum.set_index(['workload','run_id'])
    .style
    .format({'pg_records':'{:,.0f}','batch_tp_rps':'{:,.0f}',
             'lat_avg_s':'{:.1f}','lat_p50_s':'{:.1f}','lat_p95_s':'{:.1f}',
             'stream_recv':'{:,.0f}','stream_influx':'{:,.0f}','stream_rps':'{:.1f}','stream_loss_pct':'{:.1f}%'})
    .background_gradient(subset=['stream_loss_pct'], cmap='Reds')
)

⚠  4x R1 (jun-10): pg_records contém dados de run anterior (sem truncate).
   Throughput batch e docker stats são válidos; pg_records NÃO comparável.



---
## 2. Consistência Intra-Workload

Verifica se as repetições de cada workload produziram resultados estáveis.  
**CV ≤ 10%** é considerado aceitável em benchmarks de sistemas.

In [4]:
# ── Coeficiente de Variação ───────────────────────────────────────────────────
CV_METRICS = {
    'batch_tp_rps':    'Throughput batch (rec/s)',
    'lat_avg_s':       'Latência média (s)',
    'lat_p50_s':       'Latência p50 (s)',
    'lat_p95_s':       'Latência p95 (s)',
    'stream_loss_pct': 'Perda stream (%)',
}

cv_rows = []
for wl, grp in df_sum.groupby('workload'):
    for col, label in CV_METRICS.items():
        vals = grp[col].dropna()
        mean = vals.mean()
        std  = vals.std() if len(vals) > 1 else 0
        cv   = std / mean * 100 if mean else np.nan
        cv_rows.append({'Workload': wl, 'Métrica': label,
                        'Média': mean, 'DP': std, 'CV (%)': cv})

df_cv = pd.DataFrame(cv_rows)
cv_pivot = df_cv.pivot(index='Métrica', columns='Workload', values='CV (%)')

print('Coeficiente de Variação — CV ≤ 10% indica alta consistência\n')
display(
    cv_pivot.style
    .format('{:.1f}%', na_rep='—')
    .background_gradient(cmap='RdYlGn_r', vmin=0, vmax=15)
)

Coeficiente de Variação — CV ≤ 10% indica alta consistência



Workload,1x,2x,4x
Métrica,,,
Latência média (s),2.4%,1.5%,1.1%
Latência p50 (s),2.7%,2.1%,1.4%
Latência p95 (s),1.1%,1.4%,0.1%
Perda stream (%),—,—,—
Throughput batch (rec/s),13.2%,11.8%,9.5%


In [ ]:
# ── Consistência intra-workload: throughput batch (boxplot + strip) ───────────
# Exclui o primeiro e o último ciclo de cada run (efeito de borda experimental):
# o primeiro ciclo processa um buffer parcialmente preenchido (experimento ainda
# acelerando); o último processa o resíduo do buffer no momento em que o gerador
# encerra, podendo conter apenas alguns registros.
from datetime import datetime as _dt

def _trim_edges(df):
    frames = []
    for (wl, rid), grp in df.groupby(['workload', 'run_id']):
        grp = grp.copy()
        grp['_ts'] = (grp['batch_id']
                      .str.replace('batch_', '', regex=False)
                      .apply(lambda x: _dt.strptime(x, '%Y%m%d_%H%M%S')))
        grp = grp.sort_values('_ts')
        frames.append(grp.iloc[1:-1])
    return pd.concat(frames, ignore_index=True) if frames else df.iloc[0:0]

df_tp_trimmed = _trim_edges(df_tp)

fig, axes = plt.subplots(1, 3, figsize=FIG2T, sharey=False)

for ax, wl in zip(axes, WORKLOADS):
    grp  = df_tp_trimmed[df_tp_trimmed['workload'] == wl]
    runs = sorted(grp['run_id'].unique())
    data = [grp[grp['run_id'] == r]['throughput_records_per_s'].values for r in runs]

    bp = ax.boxplot(data, patch_artist=True,
                    medianprops=dict(color='black', linewidth=1.5),
                    flierprops=dict(marker='o', markersize=2.5, alpha=0.4))
    for patch in bp['boxes']:
        patch.set(facecolor=C[wl], alpha=0.55)

    for i, (r, d) in enumerate(zip(runs, data), start=1):
        jitter = np.random.uniform(-0.08, 0.08, size=len(d))
        ax.scatter(i + jitter, d, color=C[wl], s=8, alpha=0.4, zorder=3)

    ax.set_xticks(range(1, len(runs) + 1))
    ax.set_xticklabels(runs)
    ax.set_title(f'Workload {wl}', fontweight='bold')
    ax.set_xlabel('Execução')
    if ax is axes[0]:
        ax.set_ylabel('Throughput batch (rec/s)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v/1000:.0f}k'))

fig.suptitle('Consistência Intra-workload — Throughput Batch por Execução',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/01_consistencia_throughput_batch.png', dpi=300)
plt.show()


**Nota metodológica — Tratamento dos ciclos de borda no throughput batch**

No cálculo do boxplot acima foram desconsiderados o primeiro e o último ciclo de processamento de cada execução. O primeiro ciclo opera sobre um buffer parcialmente preenchido, pois o gerador de sensores ainda está acelerando no início do experimento — o volume de registros disponíveis é inferior ao de um ciclo estável, o que infla artificialmente o tempo relativo de inserção e reduz o throughput calculado. O último ciclo, por sua vez, processa apenas o resíduo acumulado no buffer no instante em que o gerador encerra: em experimentos de 1x foram observados ciclos finais com 3 a 44 registros (contra ~4.800 dos ciclos regulares), resultando em valores de throughput ordens de grandeza abaixo da mediana. Ambos os casos são consequências diretas do design experimental — início e fim do gerador em tempo fixo — e não refletem o comportamento estacionário do pipeline.

Também foi perceptível uma variação crescente no comportamento do PostgreSQL sob maior carga de trabalho. Nos experimentos de carga 4x, o throughput de inserção oscilou entre ~8.000 e ~20.000 rec/s dentro de um mesmo run, um spread de 2,5× em torno da mediana. Essa instabilidade não é ruído aleatório: reflete a pressão acumulada de checkpoints do WAL (*Write-Ahead Log*), ciclos de autovacuum disparados pela alta taxa de escrita e contenção de páginas de índice B-tree sob inserção contínua. O fenômeno evidencia que o modelo batch com PostgreSQL não escala de forma linear com o volume de dados — comportamento relevante para a análise comparativa proposta neste trabalho.


In [ ]:

# ── Consistência Docker: CPU e Memória por run ─────────────────────────────────
CTRS = ['batch_consumer', 'stream_consumer', 'mosquitto', 'postgres', 'influxdb']

df_dagg = (
    df_dock[df_dock['name'].isin(CTRS)]
    .groupby(['workload','run_id','name'])[['cpu_pct','mem_mib']]
    .mean().reset_index()
)

run_colors = [C['1x'], C['2x'], C['4x']]

for metric, ylabel, title_sfx in [
    ('cpu_pct', 'CPU médio (%)',       'CPU Médio'),
    ('mem_mib', 'Memória média (MiB)', 'Memória Média'),
]:
    fig, axes = plt.subplots(1, 3, figsize=FIG2T, sharey=False)
    for ax, wl in zip(axes, WORKLOADS):
        grp   = df_dagg[df_dagg['workload'] == wl]
        pivot = grp.pivot(index='name', columns='run_id', values=metric).reindex(CTRS).dropna(how='all')
        if pivot.empty: continue
        pivot.plot(kind='bar', ax=ax, color=run_colors[:len(pivot.columns)],
                   alpha=0.85, edgecolor='white', width=0.7)
        ax.set_title(f'Workload {wl}', fontweight='bold')
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=30)
        if ax is axes[0]:
            ax.set_ylabel(ylabel)
        ax.legend(title='Run', fontsize=7)

    fig.suptitle(f'Consistência Intra-workload — {title_sfx} por Container',
                 fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(f'{FIGS_DIR}/02_consistencia_{metric}.png', dpi=300)
    plt.show()


---
## 3. Comparativo entre Workloads

Cada gráfico mostra **batch (esquerda) × stream (direita)**, com barras agrupadas por workload (1x, 2x, 4x).  
Valores representam **média ± desvio padrão** entre execuções válidas.

In [8]:
# ── Helpers ───────────────────────────────────────────────────────────────────
xi = np.arange(len(WORKLOADS))

def agg(metric):
    means, stds = [], []
    for wl in WORKLOADS:
        vals = df_sum[df_sum['workload'] == wl][metric].dropna()
        means.append(vals.mean())
        stds.append(vals.std() if len(vals) > 1 else 0)
    return np.array(means), np.array(stds)

def agg_docker(container, metric):
    means, stds = [], []
    for wl in WORKLOADS:
        per_run = (
            df_dock[(df_dock['workload']==wl) & (df_dock['name']==container)]
            .groupby('run_id')[metric].mean()
        )
        means.append(per_run.mean())
        stds.append(per_run.std() if len(per_run) > 1 else 0)
    return np.array(means), np.array(stds)

def side_bars(ax, means, stds, ylabel='', title='', color_by_workload=True):
    colors = [C[w] for w in WORKLOADS] if color_by_workload else [C['batch']]*3
    bars = ax.bar(xi, means, yerr=stds, capsize=4, width=0.55,
                  color=colors, alpha=0.85,
                  error_kw=dict(elinewidth=1.2, ecolor='#333'),
                  edgecolor='#333', linewidth=0.5)
    ax.set_xticks(xi)
    ax.set_xticklabels(WORKLOADS)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    return bars

def annot_bars(ax, means, stds, fmt):
    for i, (m, s) in enumerate(zip(means, stds)):
        if np.isnan(m): continue
        ax.text(i, m + (s or 0) + m*0.02, fmt(m),
                ha='center', fontsize=7)

print('Helpers OK')

Helpers OK


In [ ]:
# ── 3.1 Throughput de Ingestão — burst por execução de cron e efetivo ────────
from datetime import datetime as _dt
from matplotlib.patches import Patch as _Patch

CRON_INTERVAL_S = 300  # cron executa a cada 5 minutos

def _cron_stats(wl):
    """Agrupa lotes CSV em janelas de 5 min (uma execução de cron) e retorna
    throughput burst e registros medianos por execução."""
    tp_list, rec_list = [], []
    for rid, sub in df_tp[df_tp['workload'] == wl].groupby('run_id'):
        sub = sub.copy()
        sub['_ts'] = (sub['batch_id']
                      .str.replace('batch_', '', regex=False)
                      .apply(lambda x: _dt.strptime(x, '%Y%m%d_%H%M%S')))
        sub = sub.sort_values('_ts').iloc[1:-1]  # remove bordas
        if sub.empty:
            continue
        t0 = sub['_ts'].iloc[0]
        sub['_win'] = ((sub['_ts'] - t0).dt.total_seconds() // CRON_INTERVAL_S).astype(int)
        for _, grp in sub.groupby('_win'):
            total_rec  = grp['records'].sum()
            total_secs = grp['processing_seconds'].sum()
            if total_secs > 0:
                tp_list.append(total_rec / total_secs)
            rec_list.append(total_rec)
    import statistics as _st
    return (_st.mean(tp_list)  if tp_list  else float('nan'),
            _st.stdev(tp_list) if len(tp_list) > 1 else 0,
            _st.median(rec_list) if rec_list else 0)

cron_burst_mean = np.array([_cron_stats(wl)[0] for wl in WORKLOADS])
cron_burst_std  = np.array([_cron_stats(wl)[1] for wl in WORKLOADS])
cron_rec_median = [_cron_stats(wl)[2] for wl in WORKLOADS]

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(FIG2[0], FIG2[1] * 1.55),
    gridspec_kw={'hspace': 0.6})

# ── Superior: throughput burst por execução de cron ──────────────────────────
side_bars(ax_top, cron_burst_mean, cron_burst_std, ylabel='rec/s',
          title='Throughput de Inserção por Execução de Cron — Batch (PostgreSQL)')
annot_bars(ax_top, cron_burst_mean, cron_burst_std, lambda v: f'{v/1000:.0f}k')
ax_top.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1000:.0f}k'))
ax_top.set_xticklabels(
    [f'{wl}\ncron 5 min\n~{cron_rec_median[i]/1000:.0f}k reg/exec'
     for i, wl in enumerate(WORKLOADS)],
    fontsize=8)
ax_top.margins(y=0.20)

# ── Inferior: throughput efetivo sustentado — batch ≈ stream mesma escala ────
m_eff_b, s_eff_b = agg('batch_effective_rps')
m_eff_s, s_eff_s = agg('stream_rps')

w = 0.35
ax_bot.bar(xi - w/2, m_eff_b, width=w, yerr=s_eff_b, capsize=4,
           color=[C[wl] for wl in WORKLOADS], alpha=0.85,
           edgecolor='#333', linewidth=0.5,
           error_kw=dict(elinewidth=1.2, ecolor='#333'))
ax_bot.bar(xi + w/2, m_eff_s, width=w, yerr=s_eff_s, capsize=4,
           color=[C[wl] for wl in WORKLOADS], alpha=0.45, hatch='//',
           edgecolor='#333', linewidth=0.5,
           error_kw=dict(elinewidth=1.2, ecolor='#333'))

for i, (mb, ms) in enumerate(zip(m_eff_b, m_eff_s)):
    ax_bot.text(i - w/2, mb * 1.06, f'{mb:.0f}', ha='center', fontsize=7)
    ax_bot.text(i + w/2, ms * 1.06, f'{ms:.0f}', ha='center', fontsize=7)

ax_bot.set_xticks(xi)
ax_bot.set_xticklabels(WORKLOADS)
ax_bot.set_ylabel('rec/s')
ax_bot.set_title('Throughput Efetivo Sustentado — Batch vs Stream', fontweight='bold')
ax_bot.margins(y=0.20)
ax_bot.legend(handles=[
    _Patch(facecolor='#aaa', alpha=0.85, edgecolor='#333', label='Batch (efetivo)'),
    _Patch(facecolor='#aaa', alpha=0.45, hatch='//', edgecolor='#333', label='Stream'),
], fontsize=8, loc='upper left')

fig.suptitle('Throughput de Ingestão por Workload', fontweight='bold')
plt.savefig(f'{FIGS_DIR}/03_throughput_batch_vs_stream.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:

# ── 3.2 Registros ingeridos — Batch (PG) esq × Stream (InfluxDB) dir ──────────
fig, (ax_b, ax_s) = plt.subplots(1, 2, figsize=FIG2T)

for ax, col, paradigm, ckey in [
    (ax_b, 'pg_records',    'Batch → PostgreSQL', 'batch'),
    (ax_s, 'stream_influx', 'Stream → InfluxDB',  'stream'),
]:
    m, s = agg(col)
    side_bars(ax, m, s, ylabel='Registros', title=paradigm)
    annot_bars(ax, m, s, lambda v: f'{v/1e6:.2f}M' if v >= 1e6 else f'{v/1e3:.0f}k')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda v,_: f'{v/1e6:.1f}M' if v >= 1e6 else f'{v/1e3:.0f}k'))

# Linha de referência: registros recebidos pelo stream consumer
m_recv, _ = agg('stream_recv')
ax_s.plot(xi, m_recv, 'k--', linewidth=1.0, label='Recebido MQTT')
ax_s.legend(fontsize=7)

print('⚠  4x R1: pg_records inclui dados de run anterior — barra 4x (batch) superestimada.')
fig.suptitle('Registros Ingeridos por Pipeline', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/04_registros_ingeridos.png', dpi=300)
plt.show()


In [ ]:

# ── 3.3 Latência batch (avg / p50 / p95) ─────────────────────────────────────
fig, ax = plt.subplots(figsize=FIG2)

lat_cols = [('lat_avg_s', 'Média', '#4477AA'),
            ('lat_p50_s', 'p50',   '#228833'),
            ('lat_p95_s', 'p95',   '#EE6677')]
width = 0.22
offsets = [-width, 0, width]

for (col, label, color), off in zip(lat_cols, offsets):
    m, s = agg(col)
    ax.bar(xi + off, m, width=width, label=label, color=color, alpha=0.85,
           yerr=s, capsize=3, error_kw=dict(elinewidth=1),
           edgecolor='#333', linewidth=0.4)
    for i, (mv, sv) in enumerate(zip(m, s)):
        ax.text(i + off, mv + (sv or 0) + mv * 0.02, f'{mv:.0f}s',
                ha='center', fontsize=6.5)

ax.set_xticks(xi)
ax.set_xticklabels(WORKLOADS)
ax.set_ylabel('Latência (s)')
ax.set_title('Latência Batch — avg / p50 / p95 (PostgreSQL)', fontweight='bold')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/05_latencia_batch.png', dpi=300)
plt.show()


### 3.4 Distribuição de Latência — Batch (s) vs. Stream (ms)

Queries diretas ao PostgreSQL e InfluxDB para obter os valores brutos de latência.  
**Batch:** `ingested_at − created_at` por registro no PG (escala em segundos).  
**Stream:** `ingested_at − created_at` por ponto no InfluxDB (escala em milissegundos).  
Valores negativos de stream (0.01%) são excluídos por serem artefato de arredondamento inteiro.

In [ ]:

# ── 3.4 Distribuição de Latência — Batch (esq, segundos) × Stream (dir, ms) ──
# Carrega histogramas pré-computados pelo export_experiment_metrics.py.
# Usa o run mais recente de cada workload que tenha o arquivo latency_hist_*.json.

def load_hist(workload):
    """Retorna o histograma do run mais recente para o workload dado."""
    pattern = f'{RUNS_DIR}/experimento_{workload}_*/metrics/latency_hist_{workload}.json'
    files = sorted(glob.glob(pattern))
    if not files:
        return None
    with open(files[-1], encoding='utf-8') as f:
        return json.load(f)

hist_data = {wl: load_hist(wl) for wl in WORKLOADS}

for wl, h in hist_data.items():
    if h is None:
        print(f'  ⚠️  {wl}: latency_hist_{wl}.json não encontrado — rode o export após o experimento')
    else:
        b = h.get('batch', {})
        s = h.get('stream', {})
        print(f'  {wl}  batch: {b.get("total_in_range",0):,} registros | '
              f'stream: {s.get("total_positive",0):,} pontos')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, (ax_b, ax_s) = plt.subplots(1, 2, figsize=FIG2T)

for wl in WORKLOADS:
    h = hist_data.get(wl)
    if not h:
        continue

    # Esquerda — Batch (segundos)
    bh = h.get('batch', {})
    if bh and 'bin_edges_s' in bh and not bh.get('error'):
        edges  = bh['bin_edges_s']
        counts = bh['counts']
        total  = bh['total_in_range'] or 1
        bw     = edges[1] - edges[0]
        density = [c / total / bw for c in counts]
        ax_b.stairs(density, edges, fill=True, alpha=0.45, color=C[wl], label=wl)
        ax_b.stairs(density, edges, fill=False, alpha=0.9, color=C[wl], linewidth=1.2)

    # Direita — Stream (ms)
    sh = h.get('stream', {})
    if sh and 'bin_edges_ms' in sh and not sh.get('error'):
        edges  = sh['bin_edges_ms']
        counts = sh['counts']
        total  = sh['total_positive'] or 1
        # bw robusto: usa diferença mediana entre edges consecutivos (evita zeros do Flux)
        diffs = [edges[i+1] - edges[i] for i in range(len(edges)-1) if edges[i+1] > edges[i]]
        bw    = sorted(diffs)[len(diffs)//2] if diffs else 1.0
        # filtra bins com largura zero antes de plotar
        valid  = [(e0, e1, c) for e0, e1, c in zip(edges[:-1], edges[1:], counts) if e1 > e0]
        if valid:
            e0s, e1s, cs = zip(*valid)
            plot_edges   = list(e0s) + [e1s[-1]]
            density      = [c / total / bw for c in cs]
            ax_s.stairs(density, plot_edges, fill=True, alpha=0.45, color=C[wl], label=wl)
            ax_s.stairs(density, plot_edges, fill=False, alpha=0.9, color=C[wl], linewidth=1.2)

ax_b.set_xlabel('Latência (s)')
ax_b.set_ylabel('Densidade')
ax_b.set_title('Batch → PostgreSQL', fontweight='bold')
ax_b.legend(title='Workload', fontsize=7)
ax_b.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.0f}s'))

stream_clip = (hist_data.get('1x') or {}).get('stream', {}).get('range_ms', [0, 30])[1]
ax_s.set_xlabel(f'Latência (ms)  [0–{stream_clip}ms, ~99.9% dos dados]')
ax_s.set_ylabel('Densidade')
ax_s.set_title('Stream → InfluxDB', fontweight='bold')
ax_s.legend(title='Workload', fontsize=7)
ax_s.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.0f}ms'))

fig.suptitle('Distribuição de Latência de Ingestão — Batch vs. Stream', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/05b_distribuicao_latencia.png', dpi=300)
plt.show()


In [ ]:

# ── 3.4 CPU médio — Batch consumer (esq) × Stream consumer (dir) ─────────────
fig, (ax_b, ax_s) = plt.subplots(1, 2, figsize=FIG2, sharey=False)

m_bc, s_bc = agg_docker('batch_consumer',  'cpu_pct')
m_sc, s_sc = agg_docker('stream_consumer', 'cpu_pct')

side_bars(ax_b, m_bc, s_bc, ylabel='CPU médio (%)', title='Batch Consumer')
annot_bars(ax_b, m_bc, s_bc, lambda v: f'{v:.1f}%')

side_bars(ax_s, m_sc, s_sc, ylabel='CPU médio (%)', title='Stream Consumer')
annot_bars(ax_s, m_sc, s_sc, lambda v: f'{v:.1f}%')

ymax = max(ax_b.get_ylim()[1], ax_s.get_ylim()[1]) * 1.1
ax_b.set_ylim(0, ymax)
ax_s.set_ylim(0, ymax)

fig.suptitle('Utilização de CPU por Workload — Consumers', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/06_cpu_consumers.png', dpi=300)
plt.show()


In [ ]:

# ── 3.5 Memória média — Batch consumer (esq) × Stream consumer (dir) ──────────
fig, (ax_b, ax_s) = plt.subplots(1, 2, figsize=FIG2, sharey=False)

m_bc, s_bc = agg_docker('batch_consumer',  'mem_mib')
m_sc, s_sc = agg_docker('stream_consumer', 'mem_mib')

side_bars(ax_b, m_bc, s_bc, ylabel='Memória média (MiB)', title='Batch Consumer')
annot_bars(ax_b, m_bc, s_bc, lambda v: f'{v:.0f} MiB')

side_bars(ax_s, m_sc, s_sc, ylabel='Memória média (MiB)', title='Stream Consumer')
annot_bars(ax_s, m_sc, s_sc, lambda v: f'{v:.0f} MiB')

ymax = max(ax_b.get_ylim()[1], ax_s.get_ylim()[1]) * 1.1
ax_b.set_ylim(0, ymax)
ax_s.set_ylim(0, ymax)

fig.suptitle('Utilização de Memória por Workload — Consumers', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/07_memoria_consumers.png', dpi=300)
plt.show()


In [ ]:

# ── 3.6 CPU infraestrutura — Mosquitto (esq) × Banco de dados (dir) ───────────
fig, axes = plt.subplots(1, 3, figsize=FIG2T, sharey=False)

containers_infra = [
    ('mosquitto', 'Mosquitto (broker MQTT)', axes[0]),
    ('postgres',  'PostgreSQL (batch)',       axes[1]),
    ('influxdb',  'InfluxDB (stream)',        axes[2]),
]

for ctr, title, ax in containers_infra:
    m, s = agg_docker(ctr, 'cpu_pct')
    side_bars(ax, m, s, ylabel='CPU médio (%)', title=title)
    annot_bars(ax, m, s, lambda v: f'{v:.1f}%')

fig.suptitle('CPU Médio — Infraestrutura por Workload', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/08_cpu_infraestrutura.png', dpi=300)
plt.show()


In [ ]:

# ── 3.7 Memória infraestrutura — escala linear lado a lado ────────────────────
fig, axes = plt.subplots(1, 3, figsize=FIG2T, sharey=False)

containers_infra_mem = [
    ('mosquitto', 'Mosquitto (broker MQTT)', axes[0]),
    ('postgres',  'PostgreSQL (batch)',       axes[1]),
    ('influxdb',  'InfluxDB (stream)',        axes[2]),
]

for ctr, title, ax in containers_infra_mem:
    m, s = agg_docker(ctr, 'mem_mib')
    side_bars(ax, m, s, ylabel='Memória média (MiB)', title=title)
    annot_bars(ax, m, s, lambda v: f'{v:.0f}')

fig.suptitle('Memória Média — Infraestrutura por Workload', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/09_memoria_infraestrutura.png', dpi=300)
plt.show()


In [ ]:

# ── 3.8 CPU temporal — Batch consumer (esq) × Stream consumer (dir) ───────────
fig, (ax_b, ax_s) = plt.subplots(1, 2, figsize=FIG2T, sharey=False)

for ax, ctr, title in [
    (ax_b, 'batch_consumer',  'Batch Consumer'),
    (ax_s, 'stream_consumer', 'Stream Consumer'),
]:
    for wl in WORKLOADS:
        sub = df_dock[(df_dock['name']==ctr) & (df_dock['workload']==wl)].copy()
        if sub.empty: continue
        sub['t_bin'] = sub['t_min'].round(0)
        ts  = sub.groupby('t_bin')['cpu_pct'].mean().rolling(3, min_periods=1).mean()
        std = sub.groupby('t_bin')['cpu_pct'].std().fillna(0).rolling(3, min_periods=1).mean()
        ax.plot(ts.index, ts.values, label=wl, color=C[wl], linewidth=1.2)
        ax.fill_between(ts.index, (ts - std).clip(0), ts + std, alpha=0.12, color=C[wl])
    ax.set_ylabel('CPU (%)')
    ax.set_xlabel('Tempo (min)')
    ax.set_title(title, fontweight='bold')
    ax.legend(title='Workload', fontsize=7)

fig.suptitle('CPU ao Longo do Tempo — média ± DP entre runs (janela móvel=3)',
             fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/10_cpu_temporal_lado_a_lado.png', dpi=300)
plt.show()


In [ ]:

# ── 3.9 Memória temporal — Batch consumer (esq) × Stream consumer (dir) ───────
fig, (ax_b, ax_s) = plt.subplots(1, 2, figsize=FIG2T, sharey=False)

for ax, ctr, title in [
    (ax_b, 'batch_consumer',  'Batch Consumer'),
    (ax_s, 'stream_consumer', 'Stream Consumer'),
]:
    for wl in WORKLOADS:
        sub = df_dock[(df_dock['name']==ctr) & (df_dock['workload']==wl)].copy()
        if sub.empty: continue
        sub['t_bin'] = sub['t_min'].round(0)
        ts  = sub.groupby('t_bin')['mem_mib'].mean().rolling(3, min_periods=1).mean()
        std = sub.groupby('t_bin')['mem_mib'].std().fillna(0).rolling(3, min_periods=1).mean()
        ax.plot(ts.index, ts.values, label=wl, color=C[wl], linewidth=1.2)
        ax.fill_between(ts.index, (ts - std).clip(0), ts + std, alpha=0.12, color=C[wl])
    ax.set_ylabel('Memória (MiB)')
    ax.set_xlabel('Tempo (min)')
    ax.set_title(title, fontweight='bold')
    ax.legend(title='Workload', fontsize=7)

fig.suptitle('Memória ao Longo do Tempo — média ± DP entre runs (janela móvel=3)',
             fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/11_memoria_temporal_lado_a_lado.png', dpi=300)
plt.show()


---
## 4. Painel Comparativo Final

Painel 2×2 para artigo: coluna esquerda = batch, coluna direita = stream.

In [ ]:

# ── Painel 2×2 — batch (col esq) × stream (col dir) ──────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(7.16, 5.5))

# (A) Throughput batch
m, s = agg('batch_tp_rps')
side_bars(axes[0,0], m, s, ylabel='rec/s', title='(A) Throughput — Batch (PostgreSQL)')
annot_bars(axes[0,0], m, s, lambda v: f'{v/1000:.0f}k')
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v/1000:.0f}k'))

# (B) Throughput stream — média de registros ingeridos por segundo
m, s = agg('stream_rps')
side_bars(axes[0,1], m, s, ylabel='rec/s', title='(B) Throughput — Stream (InfluxDB)')
annot_bars(axes[0,1], m, s, lambda v: f'{v:.0f}')

# (C) CPU — Batch consumer
m_bc, s_bc = agg_docker('batch_consumer', 'cpu_pct')
m_sc, s_sc = agg_docker('stream_consumer', 'cpu_pct')
side_bars(axes[1,0], m_bc, s_bc, ylabel='CPU médio (%)', title='(C) CPU — Batch Consumer')
annot_bars(axes[1,0], m_bc, s_bc, lambda v: f'{v:.1f}%')

# (D) CPU — Stream consumer
side_bars(axes[1,1], m_sc, s_sc, ylabel='CPU médio (%)', title='(D) CPU — Stream Consumer')
annot_bars(axes[1,1], m_sc, s_sc, lambda v: f'{v:.1f}%')

ymax_cpu = max(axes[1,0].get_ylim()[1], axes[1,1].get_ylim()[1]) * 1.15
axes[1,0].set_ylim(0, ymax_cpu)
axes[1,1].set_ylim(0, ymax_cpu)

fig.suptitle('Comparativo Batch vs. Stream — Métricas Principais', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/12_painel_comparativo_v2.png', dpi=300)
plt.show()


In [ ]:

# ── Painel de memória 1×2 — Batch consumer (esq) × Stream consumer (dir) ──────
fig, (ax_b, ax_s) = plt.subplots(1, 2, figsize=FIG2)

m_bc, s_bc = agg_docker('batch_consumer',  'mem_mib')
m_sc, s_sc = agg_docker('stream_consumer', 'mem_mib')

side_bars(ax_b, m_bc, s_bc, ylabel='Memória média (MiB)', title='(E) Memória — Batch Consumer')
annot_bars(ax_b, m_bc, s_bc, lambda v: f'{v:.0f}')

side_bars(ax_s, m_sc, s_sc, ylabel='Memória média (MiB)', title='(F) Memória — Stream Consumer')
annot_bars(ax_s, m_sc, s_sc, lambda v: f'{v:.0f}')

ymax_mem = max(ax_b.get_ylim()[1], ax_s.get_ylim()[1]) * 1.15
ax_b.set_ylim(0, ymax_mem)
ax_s.set_ylim(0, ymax_mem)

fig.suptitle('Memória — Batch vs. Stream Consumer por Workload', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/13_painel_memoria_v2.png', dpi=300)
plt.show()


---
## 5. Tabela Consolidada para o TCC

In [20]:
def fmt_mean_std(vals):
    vals = [v for v in vals if not np.isnan(v)]
    if not vals: return '—'
    m = np.mean(vals)
    s = np.std(vals)
    return f'{m:.1f} ± {s:.1f}' if len(vals) > 1 else f'{m:.1f}'

tbl_rows = []
for wl in WORKLOADS:
    wl_s  = [s for s in summaries if s['workload']==wl]
    tp_v  = [s['batch_throughput']['avg_throughput_records_per_s'] for s in wl_s]
    la_v  = [s['postgres']['latency_avg_ms']/1000 for s in wl_s]
    p95_v = [s['postgres']['latency_p95_ms']/1000 for s in wl_s]
    lo_v  = df_sum[df_sum['workload']==wl]['stream_loss_pct'].tolist()

    cpu_b  = df_dock[(df_dock['workload']==wl)&(df_dock['name']=='batch_consumer') ]['cpu_pct'].mean()
    cpu_s  = df_dock[(df_dock['workload']==wl)&(df_dock['name']=='stream_consumer')]['cpu_pct'].mean()
    mem_b  = df_dock[(df_dock['workload']==wl)&(df_dock['name']=='batch_consumer') ]['mem_mib'].mean()
    mem_s  = df_dock[(df_dock['workload']==wl)&(df_dock['name']=='stream_consumer')]['mem_mib'].mean()

    tbl_rows.append({
        'Workload':                  wl,
        'Throughput batch (rec/s)':  fmt_mean_std([int(v) for v in tp_v]),
        'Latência avg batch (s)':    fmt_mean_std(la_v),
        'Latência p95 batch (s)':    fmt_mean_std(p95_v),
        'Perda stream (%)':          fmt_mean_std(lo_v),
        'CPU batch consumer (%)':    f'{cpu_b:.2f}',
        'CPU stream consumer (%)':   f'{cpu_s:.2f}',
        'Mem batch consumer (MiB)':  f'{mem_b:.1f}',
        'Mem stream consumer (MiB)': f'{mem_s:.1f}',
    })

df_tbl = pd.DataFrame(tbl_rows).set_index('Workload')
print('Tabela consolidada (média ± DP entre execuções)\n')
display(df_tbl.T)

Tabela consolidada (média ± DP entre execuções)



Workload,1x,2x,4x
Throughput batch (rec/s),26565.0 ± 2858.9,27925.3 ± 2693.9,25868.5 ± 1730.5
Latência avg batch (s),189.2 ± 3.8,180.5 ± 2.3,167.3 ± 1.3
Latência p95 batch (s),326.1 ± 2.9,316.7 ± 3.7,300.4 ± 0.2
Perda stream (%),0.0 ± 0.0,0.0 ± 0.0,0.0 ± 0.0
CPU batch consumer (%),1.21,1.98,3.71
CPU stream consumer (%),5.20,13.39,20.64
Mem batch consumer (MiB),23.2,28.6,33.8
Mem stream consumer (MiB),34.1,34.9,34.4
